In [ ]:
import numpy as np
import pandas as pd

ACDDS Analysis - L1-L5 Statistics (基于真实调查数据)

Step 1: 读取数据...
✓ 数据形状：2000 样本 × 26 题项
✓ 数据类型：所有题项均为整数 (Likert 1-7 分)


In [ ]:
print('ACDDS Analysis - L1-L5 Statistics (基于真实调查数据)')
print('='*60)
# 读取实际调查数据
print('\nStep 1: 读取数据...')
df = pd.read_csv('acdds_raw_data_2000_medium_alpha.csv')
print(f'✓ 数据形状：{df.shape[0]} 样本 × {df.shape[1]} 题项')
print(f'✓ 数据类型：所有题项均为整数 (Likert 1-7 分)')

In [ ]:
n = len(df)
# 维度定义（题项数、反向题索引、权重）
dimensions = {
    'PU': {'items': 5, 'reverse': [], 'w': 0.25},      # 感知有用性
    'PEOU': {'items': 4, 'reverse': [], 'w': 0.20},    # 感知易用性
    'TR': {'items': 5, 'reverse': [], 'w': 0.20},      # AI 信任
    'AA': {'items': 5, 'reverse': [1, 3], 'w': 0.15},  # 算法态度
    'AC': {'items': 4, 'reverse': [1], 'w': 0.10},     # 自主性与控制
    'SF': {'items': 3, 'reverse': [1, 2], 'w': 0.10}   # 社会影响
}

In [ ]:
# 对反向题进行正向化处理
print('\nStep 2: 反向题正向化...')
df_corr = df.copy()
for dim, info in dimensions.items():
    for idx in info['reverse']:
        col = f'{dim}{idx+1}'
        df_corr[col] = 8 - df_corr[col]
        print(f'  ✓ {col} 已正向化')


Step 2: 反向题正向化...
  ✓ AA2 已正向化
  ✓ AA4 已正向化
  ✓ AC2 已正向化
  ✓ SF2 已正向化
  ✓ SF3 已正向化


In [5]:
# Cronbach's alpha 信度分析
print('\nStep 3: 信度分析 (Cronbach\'s α)...')
def calc_alpha(d):
    k = d.shape[1]
    return (k/(k-1)) * (1 - d.var(axis=0).sum()/d.sum(axis=1).var())

alphas = {}
for dim, info in dimensions.items():
    cols = [f'{dim}{i+1}' for i in range(info['items'])]
    alpha = calc_alpha(df_corr[cols])
    alphas[dim] = alpha
    quality = '优秀' if alpha > 0.8 else '良好' if alpha > 0.7 else '不达标'
    print(f'  {dim}: α = {alpha:.4f} ({quality})')


Step 3: 信度分析 (Cronbach's α)...
  PU: α = 0.8860 (优秀)
  PEOU: α = 0.8354 (优秀)
  TR: α = 0.8938 (优秀)
  AA: α = 0.8640 (优秀)
  AC: α = 0.8221 (优秀)
  SF: α = 0.7996 (良好)


In [ ]:
# 总问卷信度
total_cols = list(df_corr.columns)
total_alpha = calc_alpha(df_corr)
quality = '优秀' if total_alpha > 0.8 else '良好' if total_alpha > 0.7 else '不达标'
print(f'  总问卷：α = {total_alpha:.4f} ({quality})')

  总问卷：α = 0.8973 (优秀)


In [11]:
# 熵权法计算客观权重
print('\nStep 4: 熵权法计算客观权重...')
dim_means = {}
for dim, info in dimensions.items():
    cols = [f'{dim}{i+1}' for i in range(info['items'])]
    dim_means[dim] = df_corr[cols].mean(axis=1)

dm = pd.DataFrame(dim_means)
dn = (dm - dm.min()) / (dm.max() - dm.min() + 1e-10)
p = dn / dn.sum()
e = -1/np.log(n) * (p * np.log(p + 1e-10)).sum()
d = 1 - e
weights = d / d.sum()

print('\n各维度熵权权重:')
for dim, w in zip(dimensions.keys(), weights):
    print(f'  {dim}: {w:.4f} ({w*100:.2f}%)')

# 综合得分计算
print('\nStep 5: 计算综合依赖等级得分...')
composite = sum(dim_means[dim] * weights[i] for i, dim in enumerate(dimensions.keys()))
normalized = (composite - composite.min()) / (composite.max() - composite.min())
normalized


Step 4: 熵权法计算客观权重...

各维度熵权权重:
  PU: 0.1688 (16.88%)
  PEOU: 0.1700 (17.00%)
  TR: 0.1985 (19.85%)
  AA: 0.1765 (17.65%)
  AC: 0.1301 (13.01%)
  SF: 0.1561 (15.61%)

Step 5: 计算综合依赖等级得分...


C:\Users\admin\AppData\Local\Temp\ipykernel_21680\533389730.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  composite = sum(dim_means[dim] * weights[i] for i, dim in enumerate(dimensions.keys()))


0       0.495172
1       0.257261
2       0.503757
3       0.621947
4       0.599822
          ...   
1995    0.527170
1996    0.466419
1997    0.926012
1998    0.011843
1999    0.469276
Length: 2000, dtype: float64

In [8]:
# L1-L5 等级划分
def get_level(s):
    if s < 0.2: return 'L1'
    elif s < 0.4: return 'L2'
    elif s < 0.6: return 'L3'
    elif s < 0.8: return 'L4'
    else: return 'L5'

levels = normalized.apply(get_level)
lc = levels.value_counts().sort_index()
lp = lc / len(levels) * 100

# 输出 L1-L5 等级分布
print('\n' + '='*60)
print('L1-L5 等级分布统计 (N=2000)')
print('='*60)
header = f'{"Level":<10}{"Count":<10}{"Percentage (%)":<15}'
print(header)
print('-'*60)
for lvl in ['L1', 'L2', 'L3', 'L4', 'L5']:
    count = lc.get(lvl, 0)
    pct = lp.get(lvl, 0)
    bar = '█' * int(pct / 2)
    line = f'{lvl:<10}{count:<10}{pct:<15.2f}{bar}'
    print(line)
total_line = f'{"Total":<10}{n:<10}{100.00:<15.2f}'
print(total_line)
print('='*60)


L1-L5 等级分布统计 (N=2000)
Level     Count     Percentage (%) 
------------------------------------------------------------
L1        75        3.75           █
L2        522       26.10          █████████████
L3        919       45.95          ██████████████████████
L4        430       21.50          ██████████
L5        54        2.70           █
Total     2000      100.00         


In [9]:
# 保存结果
print('\nStep 6: 保存分析结果...')
res = pd.DataFrame({
    'ID': range(1, n+1),
    'Composite_Score': composite,
    'Normalized_Score': normalized,
    'Dependency_Level': levels
})
res.to_csv('acdds_results.csv', index=False, encoding='utf-8-sig')
print(f'✓ 结果已保存至：acdds_results.csv')

# 附加统计信息
print('\n' + '='*60)
print('附加统计信息')
print('='*60)
print(f'\n综合得分描述统计:')
print(composite.describe().round(2))
print(f'\n标准化得分描述统计:')
print(normalized.describe().round(2))

# 各维度均值
print(f'\n各维度均值:')
for dim in dimensions.keys():
    print(f'  {dim}: {dim_means[dim].mean():.2f} ± {dim_means[dim].std():.2f}')

print('\n' + '='*60)
print('✓ 分析完成！')
print('='*60)


Step 6: 保存分析结果...
✓ 结果已保存至：acdds_results.csv

附加统计信息

综合得分描述统计:
count    2000.00
mean        3.96
std         0.64
min         2.02
25%         3.52
50%         3.96
75%         4.39
max         6.00
dtype: float64

标准化得分描述统计:
count    2000.00
mean        0.49
std         0.16
min         0.00
25%         0.38
50%         0.49
75%         0.60
max         1.00
dtype: float64

各维度均值:
  PU: 4.08 ± 1.05
  PEOU: 3.85 ± 0.98
  TR: 3.95 ± 1.09
  AA: 3.77 ± 0.97
  AC: 4.15 ± 0.95
  SF: 4.00 ± 0.98

✓ 分析完成！
